In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [5]:
if is_main:
    seed = 44
    environment_string = "pong"
    gold_timesteps = 15_000_000
    training_timesteps = 500_000
    num_concepts_selected = 57
    out_folder = "basic"
    method = "lp" 


In [6]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [9]:
subset, idx = policy_coverage_selection_lp(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,rollout_steps=10_000)

There are 80000 observations
Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
Coverage 0.99605


In [7]:
num_pairs_lp=20_000
rollout_steps=10_000

unique_actions = list(set([int(i[1]) for i in q_estimates]))
actions = np.array([i[1] for i in q_estimates])

# Continuous
discretized_X = np.array([i[0] for i in q_estimates])

q_values = np.array([i[2] for i in q_estimates])

final_vals = []
num_actions = len(set([i[1] for i in q_estimates]))
print(num_actions)
seen = set() 
for a in unique_actions:
        relevant_idx = np.where(actions == a)[0]
        if len(relevant_idx) <= 500:
            relevant_low = relevant_high = relevant_idx
        else:
            relevant_low = np.argsort(np.abs(q_values))[:500]
            relevant_high = np.argsort(np.abs(q_values))[-500:]
        for low_idx in relevant_low:
            for high_idx in relevant_high:
                diff = abs(q_values[low_idx] - q_values[high_idx])
                # tuple of differing concept indices
                diffs = tuple(i for i, (l, h) in enumerate(zip(discretized_X[low_idx], discretized_X[high_idx])) if l != h)
                tup = (diff, diffs)
                if diffs not in seen and diffs != ():
                    seen.add(diffs)
                    final_vals.append(tup)
final_vals = final_vals[:250_000]
final_vals = sorted(final_vals,reverse=True)
weights = [i[0] for i in final_vals]

# --------------------------------------------------
# Collect observations / actions (same as before)
# --------------------------------------------------
all_observations = []
all_actions = []

obs, info = ground_truth_gym_env.reset()

for _ in range(rollout_steps):
    actions = groundtruth_model.predict(obs)[0]
    for j in range(len(actions)):
        all_observations.append([c(info[j]['observation']) for c in concept_list])
        all_actions.append(actions[j])
    obs, rew, t_1, t_2, info = ground_truth_gym_env.step(actions)

all_observations = np.asarray(all_observations, dtype=np.int8)
all_actions = np.asarray(all_actions)

N, K = all_observations.shape

print("There are {} observations".format(N))

# --------------------------------------------------
# Sample cross-action pairs
# --------------------------------------------------
idx_i = np.random.randint(low=0, high=N, size=5 * num_pairs_lp)
idx_j = np.random.randint(low=0, high=N, size=5 * num_pairs_lp)

valid = all_actions[idx_i] != all_actions[idx_j]
idx_i = idx_i[valid][:num_pairs_lp]
idx_j = idx_j[valid][:num_pairs_lp]

if len(idx_i) == 0:
    raise ValueError("No cross-action pairs sampled.")

M = len(idx_i)

disagreement = (all_observations[idx_i] != all_observations[idx_j]).astype(np.int8)


6
There are 80000 observations


In [12]:
model = gp.Model("max_coverage_lp")
model.Params.OutputFlag = 0

ub = 1.0
len_x_vals = 0
trials = 0

# x_d variables (concept selection)
x = model.addVars(K, lb=0.0, ub=1.0, vtype=GRB.BINARY, name="x")

# y_p variables (pair covered)
y = model.addVars(M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="y")

y_2 = model.addVars(len(final_vals), lb=0.0, ub=ub, vtype=GRB.BINARY, name="y_2")

for i, (_, elems) in enumerate(final_vals):
    if elems:  # make sure not empty
        model.addConstr(
            y_2[i] <= gp.quicksum(x[e] for e in elems),
            name=f"cover_{i}"
        )
    else:
        model.addConstr(y_2[i] == 0)  # cannot be covered

# Prefix constraints: enforce consecutive coverage
# y[i] <= y[i-1] for i>0
for i in range(1, len(final_vals)):
    model.addConstr(y_2[i] <= y_2[i-1], name=f"prefix_{i}")

for p in range(M):
    model.addConstr(
        y[p] <= gp.quicksum(disagreement[p, d] * x[d] for d in range(K)),
        name=f"cover_{p}",
    )

# Cardinality constraint
model.addConstr(
    gp.quicksum(x[d] for d in range(K)) <= num_concepts_selected,
    name="budget",
)

# Constraint: maximize covered pairs
model.addConstr(gp.quicksum(y[p] for p in range(M))/M >= 0.95)
model.setObjective(gp.quicksum(weights[i]*y_2[i] for i in range(len(final_vals))), GRB.MAXIMIZE)    
model.optimize()

In [13]:
x_vals = np.array([x[d].X for d in range(K)])
y_vals = np.array([y[p].X for p in range(M)])
len_x_vals = sum(x_vals)


print("There are {} x vals".format(len_x_vals))
idx = [i for i in range(len(x_vals)) if x_vals[i] > 0.5]


There are 53.0 x vals


In [14]:
np.mean([np.sum(i*x_vals)>0 for i in disagreement] )

0.99495

In [15]:
fixed_idx = deepcopy(idx)

In [19]:
model = gp.Model("max_coverage_lp")
model.Params.OutputFlag = 0

ub = 1.0
len_x_vals = 0
trials = 0

# x_d variables (concept selection)
x = model.addVars(K, lb=0.0, ub=1.0, vtype=GRB.BINARY, name="x")

# y_p variables (pair covered)
y = model.addVars(M, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="y")

y_2 = model.addVars(len(final_vals), lb=0.0, ub=ub, vtype=GRB.BINARY, name="y_2")

for i, (_, elems) in enumerate(final_vals):
    if elems:  # make sure not empty
        model.addConstr(
            y_2[i] <= gp.quicksum(x[e] for e in elems),
            name=f"cover_{i}"
        )
    else:
        model.addConstr(y_2[i] == 0)  # cannot be covered

# Prefix constraints: enforce consecutive coverage
# y[i] <= y[i-1] for i>0
for i in range(1, len(final_vals)):
    model.addConstr(y_2[i] <= y_2[i-1], name=f"prefix_{i}")

for p in range(M):
    model.addConstr(
        y[p] <= gp.quicksum(disagreement[p, d] * x[d] for d in range(K)),
        name=f"cover_{p}",
    )

# Cardinality constraint
model.addConstr(
    gp.quicksum(x[d] for d in range(K)) <= num_concepts_selected,
    name="budget",
)

# for i in fixed_idx:
#     model.addConstr(x[i] == 1)

# Constraint: maximize covered pairs
model.setObjective(gp.quicksum(y[p] for p in range(M)), GRB.MAXIMIZE)    
model.optimize()

In [21]:
x_vals = np.array([x[d].X for d in range(K)])
y_vals = np.array([y[p].X for p in range(M)])
len_x_vals = sum(x_vals)


print("There are {} x vals".format(len_x_vals))
idx = [i for i in range(len(x_vals)) if x_vals[i] > 0.5]
np.mean([np.sum(i*x_vals)>0 for i in disagreement] )

There are 44.0 x vals


0.99575

In [52]:
subset, idx_new = policy_coverage_selection_lp_hybrid(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    q_estimates,rollout_steps=10_000)

6
There are 80000 observations
There are 51.0 x vals
6
There are 8000 observations
There are 57.0 x vals
Coverage 0.99815


In [57]:
len(set(idx_new).intersection(set(idx_original)))/len(idx_original)

0.8421052631578947

In [40]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Optional but recommended for determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
model_params = get_model(environment_string,"MlpPolicy",custom_name="cart_pole_extension")
model_params['env'] = environment_string
ground_truth_env.seed(seed)


name = "cart_pole_extension"
groundtruth_model.env = ground_truth_env

In [47]:
subset_concept, idx = policy_coverage_selection_lp_hybrid(ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    q_estimates)

2
There are 8000 observations
Set parameter Username
Set parameter LicenseID to value 2709943
Academic license - for non-commercial use only - expires 2026-09-17
2
There are 8000 observations
2
There are 8000 observations
There are 3.0 x vals
Coverage 0.85545


In [48]:
idx 

[5, 6, 7]

In [64]:
evaluate_model(environment_string,ground_truth_gym_env,groundtruth_model,seed)

498.61

In [8]:
subset_concept, idx = random_selection(concept_list,num_concepts_selected)

In [9]:
training_timesteps = 100_000


In [13]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,subset_concept,seed,processed_concepts=processed_concepts,concept_idx=idx)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_perfect_{}_{}".format(environment_string,method,seed),seed=seed)    

approx_kl,▁▁▁▁▁▁▁▁▃▃▂▂▂▁▁▁▁▃▃▃▃▃▂▂▂▂▂▆▆▃▃▅▅▅▅▆▆▆▆█
clip_fraction,▁▁▁▁▁▁▁▁▁▁▄▄▄▁▁▁▁▁▁▁▁▁▁▁▁▆▆▁▅▃▃▃▃▃██████
ema_norm_reward,▂▁▁▂▃▂▂▄▃▂▂▂▂▂▁▃▃▃▃▂▂▂▃▂▂▂▂▂▅▆▃▅▇▆▅▇▆▇██
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅███
episode_length_mean,█▄███████▃█████████████████▁██████▄██▆▅█
episode_reward_max,▁▁▁▁▅▁▁▁▁▇▁▁▁▃▂▄▁▁▁▁▁▇▁▁▁▄▁▁▁▁▇▄▁▄▁▇▁▁█▁
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁█▁▁▁▁▂▁▁▄▁▁▃▂▃
episode_reward_min,▁▁▁▁▇▄▁▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂▂▁▁▂▆▁▁▃▇▁▇█▁▁▂█▁
episodes_completed,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
explained_variance,▁▁▁███████▇▇▇▆▆▆▅▅▅▅▆▆▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
+1,...


In [11]:
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if os.path.exists(model_name):
    concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
    concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
    concept_predictor.eval()

In [12]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,processed_concepts=processed_concepts)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_imperfect_{}_{}".format(environment_string,method,seed)) 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▅▅▅▅▂▇▇▇▇▇▅▅▅▅▁▁▁▄▄▄▄▅▅▅▃▃▃███▂▂▂▂▄▄▄▃▃▃
clip_fraction,▄▄▄▄▁▃▃▃▃▃▄▄▁▁▁▁▁▁▁▁▃▃▃▁██████▁▁▁▁▁▁▁▁▁▁
ema_norm_reward,▂▃▂▂▂▁▁▂▁▁▁▁▁▁▁▃▂▂▂▂▃▁▂▂▂▂▂▂▂▁▄▆▅▆▅▄▄▆▃█
entropy_loss,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███
episode_length_mean,█▇███████████████████▇▇▁███▂█▄█▅▆██▅█▃▅▅
episode_reward_max,▁▁▁▁▁▁▂▁▁▁▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▅▁▄▄▃▁▃▃▁█▁▁
episode_reward_mean,█▃▁▁▁▁▁▁▁▁▁▃▁▂▃▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▂▅▃█▄▁▁
episode_reward_min,▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▄▁▁▁▁▁▁▃▁▄▁▁▁▄▁▁▁▁▅
episodes_completed,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇██████
explained_variance,▁▁▁▁▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█████████
+1,...
